# Question-Driven EDA : Ames Housing Data

[Contents]  
- 다양한 시각화와 통계 정보를 활용하여 데이터의 특성을 이해하고, 전처리가 필요한 문제점을 직접 발견하는 것이 목표
- 각 단계에서 얻은 결과를 바탕으로 왜 이러한 현상이 나타나는지, 어떤 전처리가 필요할지 스스로 판단해 보는 것이 이번 실습의 방향성  
- EDA 결과를 정답처럼 받아들이기보다, 관찰한 근거를 바탕으로 질문에 답하는 것 권장  
  
[Data]  
Ames Housing Raw 데이터를 바탕으로 교육용으로 간소화
- Target: SalePrice
- Numeric Features: 10개
- Categorical Features: 4개

[Note]  
- 배포된 "ames_housing_edu.csv" 파일을 직접 업로드


## 1. 라이브러리 Import

- 데이터 처리: pandas, numpy
- 시각화: matplotlib, seaborn


In [1]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas version:", pd.__version__)

pandas version: 2.2.2


## 2. Config

- 실수값을 소수 둘째 자리까지 출력


In [2]:
pd.set_option("display.float_format", "{:.2f}".format)

## 3. 데이터 업로드

- 배포된 "ames_housing_edu.csv" 파일을 업로드
- 업로드한 파일을 DataFrame으로 로드


In [5]:
from google.colab import files

uploaded = files.upload()

csv_filename = "/data/ames_housing_edu.csv"
if csv_filename not in uploaded:
    raise FileNotFoundError(
        f"{csv_filename} 파일을 업로드 하세요. "
        f"업로드된 파일: {list(uploaded.keys())}"
    )

df = pd.read_csv(io.BytesIO(uploaded[csv_filename]))

print("Uploaded data shape:", df.shape)
display(df.head())

KeyboardInterrupt: 

# df세팅

In [4]:
from pathlib import Path
data_dir = Path("/data")
csv_filename = Path(data_dir) / "ames_housing_edu.csv"
df = read_csv(csv_filename)

NameError: name 'read_csv' is not defined

## 4. 데이터 구조 확인

[Question 1]  
- 데이터는 몇 개의 행과 열로 구성되어 있는가?
- Target 변수는 무엇인가?
- 수치형 변수와 범주형 변수는 각각 몇 개인가?
- 분석 전에 자료형을 변경해야 할 변수는 보이는가?


In [ ]:
print("Data shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nData samples:")
display(df.head())


In [ ]:
# 수치형 변수와 범주형 변수 구분
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Numeric variables:", len(numeric_columns), numeric_columns)
print("Categorical variables:", len(categorical_columns), categorical_columns)

## 5. 기술통계 확인

[Question 2]  
- 주택가격의 평균과 중앙값은 비슷한가?
- 평균과 중앙값의 차이는 분포의 어떤 특징을 암시하는가?
- 최소값과 최대값의 차이가 큰 변수는 무엇인가?
- 변수의 단위가 서로 다르다는 점은 이후 분석에 어떤 영향을 줄 수 있는가?


In [ ]:
# 수치형 변수 기술통계
display(df[numeric_columns].describe().T)

# 범주형 변수 기술통계
display(df[categorical_columns].describe().T)

## 6. 결측치 확인

[Question 3]  
- 결측치가 있는 변수는 무엇인가?
- 결측치는 단순한 누락일까, 특정 시설이 없다는 의미일까?
- 결측치 비율이 분석에 영향을 줄 만큼 큰가?
- 결측치 처리 전에 원래 의미를 확인해야 하는 이유는 무엇인가?


In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Ratio(%)": df.isna().mean() * 100
}).sort_values("Missing_Count", ascending=False)

missing_summary = missing_summary[missing_summary["Missing_Count"] > 0]
display(missing_summary)

## 7. Target 분포 확인

[Question 4]  
- SalePrice는 대칭적인 분포인가?
- 고가 주택이 분포의 오른쪽 꼬리를 길게 만드는가?
- 평균과 중앙값 중 대표값으로 어느 값이 더 적절해 보이는가?
- 향후 회귀분석에서 Target 변환을 고려할 필요가 있는가?


In [ ]:
target = "SalePrice"

print("Mean   :", df[target].mean())
print("Median :", df[target].median())
print("Skewness:", df[target].skew())

plt.figure(figsize=(10, 5))
sns.histplot(data=df, x=target, bins=40, kde=True)
plt.axvline(df[target].mean(), linestyle="--", label="Mean")
plt.axvline(df[target].median(), linestyle=":", label="Median")
plt.title("Distribution of SalePrice")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
sns.boxplot(data=df, x=target)
plt.title("Box Plot of SalePrice")
plt.tight_layout()
plt.show()

## 8. 수치형 변수 분포 확인

[Question 5]  
- 강한 오른쪽 왜도를 보이는 변수는 무엇인가?
- 값이 특정 구간에 집중된 변수는 무엇인가?
- 이상치 후보가 많은 변수는 무엇인가?
- 모든 수치형 변수에 동일한 전처리를 적용하는 것이 적절한가?


In [ ]:
# 수치형 설명변수 분포
numeric_features = [column for column in numeric_columns if column != target]

df[numeric_features].hist(
    bins=30,
    figsize=(16, 12),
    edgecolor="black"
)
plt.suptitle("Distribution of Numeric Features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
skewness = (
    df[numeric_features]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
    .to_frame("Skewness")
)

display(skewness)

## 9. 이상치 후보 확인

[Question 6]  
- IQR 기준으로 이상치 후보가 많은 변수는 무엇인가?
- 통계적 이상치가 반드시 오류 데이터라고 볼 수 있는가?
- Gr_Liv_Area, Lot_Area, SalePrice의 극단값은 실제 대형 주택일 가능성이 있는가?


In [ ]:
def iqr_outlier_summary(data, columns):
    results = []

    for column in columns:
        series = data[column].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = ((series < lower_bound) | (series > upper_bound)).sum()

        results.append({
            "Variable": column,
            "Lower_Bound": lower_bound,
            "Upper_Bound": upper_bound,
            "Outlier_Count": outlier_count,
            "Outlier_Ratio(%)": outlier_count / len(series) * 100
        })

    return pd.DataFrame(results).sort_values("Outlier_Count", ascending=False)

outlier_summary = iqr_outlier_summary(df, numeric_columns)
display(outlier_summary)

In [ ]:
boxplot_columns = ["SalePrice", "Gr_Liv_Area", "Lot_Area", "Total_Bsmt_SF"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for axis, column in zip(axes, boxplot_columns):
    sns.boxplot(data=df, x=column, ax=axis)
    axis.set_title(f"Box Plot of {column}")

plt.tight_layout()
plt.show()

## 10. 범주형 변수 분포 확인

[Question 7]  
- 범주 수가 가장 많은 변수는 무엇인가?
- 특정 범주에 관측값이 지나치게 집중되어 있는가?
- 표본 수가 매우 적은 범주는 향후 인코딩과 모델링에 어떤 문제를 만들 수 있는가?


In [ ]:
for column in categorical_columns:
    print(f"\n=== {column} ===")
    display(
        df[column]
        .value_counts(dropna=False)
        .to_frame("Count")
        .assign(Ratio=lambda x: x["Count"] / len(df) * 100)
    )


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for axis, column in zip(axes, categorical_columns):
    order = df[column].value_counts().index
    sns.countplot(data=df, y=column, order=order, ax=axis)
    axis.set_title(f"Count of {column}")
    axis.set_xlabel("Count")

plt.tight_layout()
plt.show()

## 11. 수치형 변수와 주택가격의 관계

[Question 8]  
- 주택가격과 가장 강한 양의 관계를 보이는 변수는 무엇인가?
- 품질 점수와 면적 중 어느 변수가 가격 차이를 더 분명하게 설명하는가?
- 상관계수가 낮아도 비선형 관계가 존재할 수 있는가?
- 상관관계만으로 인과관계를 주장할 수 있는가?


In [ ]:
correlation = (
    df[numeric_columns]
    .corr(numeric_only=True)[target]
    .sort_values(ascending=False)
    .to_frame("Correlation_with_SalePrice")
)

display(correlation)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    df[numeric_columns].corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
relationship_columns = [
    "Overall_Qual",
    "Gr_Liv_Area",
    "Year_Built",
    "Total_Bsmt_SF"
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for axis, column in zip(axes, relationship_columns):
    sns.scatterplot(data=df, x=column, y=target, alpha=0.6, ax=axis)
    sns.regplot(data=df, x=column, y=target, scatter=False, ax=axis)
    axis.set_title(f"{column} vs SalePrice")

plt.tight_layout()
plt.show()

## 12. 범주형 변수와 주택가격의 관계

[Question 9]  
- 지역별 주택가격 중앙값에 차이가 있는가?
- 주방 품질과 외장 품질이 높아질수록 가격도 높아지는가?
- 범주별 표본 수가 다를 때 박스플롯을 어떻게 해석해야 하는가?
- 범주형 변수는 향후 회귀모델에 바로 입력할 수 있는가?


In [ ]:
# Neighborhood별 주택가격 중앙값 순서
neighborhood_order = (
    df.groupby("Neighborhood", observed=False)[target]
    .median()
    .sort_values()
    .index
)

plt.figure(figsize=(12, 8))
sns.boxplot(
    data=df,
    y="Neighborhood",
    x=target,
    order=neighborhood_order
)
plt.title("SalePrice by Neighborhood")
plt.tight_layout()
plt.show()

In [ ]:
quality_order = ["Po", "Fa", "TA", "Gd", "Ex"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df,
    x="Kitchen_Qual",
    y=target,
    order=quality_order,
    ax=axes[0]
)
axes[0].set_title("SalePrice by Kitchen Quality")

sns.boxplot(
    data=df,
    x="Exter_Qual",
    y=target,
    order=quality_order,
    ax=axes[1]
)
axes[1].set_title("SalePrice by Exterior Quality")

plt.tight_layout()
plt.show()

## 13. 주요 변수 교차 탐색

[Question 10]  
- 동일한 생활면적이라도 품질에 따라 가격 차이가 나타나는가?
- Overall_Qual은 면적과 가격의 관계를 어떻게 구분하는가?
- 단일 변수만 보는 것보다 두 변수를 함께 볼 때 새롭게 발견되는 패턴은 무엇인가?


In [ ]:
plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=df,
    x="Gr_Liv_Area",
    y=target,
    hue="Overall_Qual",
    palette="viridis",
    alpha=0.7
)
plt.title("Living Area, Overall Quality and SalePrice")
plt.tight_layout()
plt.show()

## 14. EDA Findings 작성

[Report]  
아래 질문에 대한 답을 관찰 근거와 함께 작성하세요.

1. 데이터 구조에서 확인한 특징은 무엇인가?
2. 결측치가 있는 변수와 결측의 가능한 의미는 무엇인가?
3. Target 분포의 특징과 향후 고려할 변환은 무엇인가?
4. 이상치 후보가 확인된 변수와 처리 전 확인할 사항은 무엇인가?
5. 주택가격과 관계가 강해 보이는 수치형 변수는 무엇인가?
6. 가격 차이가 뚜렷하게 나타나는 범주형 변수는 무엇인가?
7. 향후 전처리 단계에서 수행해야 할 작업은 무엇인가?

[Example Format]  
- Finding:
- Evidence:
- Possible Action:
